# Hand Gesture Recognition using CNN

## Objective

The objective of this project is to build a Convolutional Neural Network (CNN) that can classify different hand gestures from images.

### Dataset
- LeapGestRecog Dataset
- 10 Hand Gesture Classes

### Libraries Used
- NumPy
- OpenCV
- Matplotlib
- TensorFlow / Keras
- Scikit-learn

In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical

import kagglehub

/Users/prakhar/CODE/Projects/SCT_ML_4/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Download Dataset

Download the LeapGestRecog dataset using KaggleHub.

In [2]:
path = kagglehub.dataset_download("gti-upm/leapgestrecog")

dataset_path = os.path.join(path, "leapGestRecog")

Resuming download from 884998144 bytes (1401086894 bytes left)...
Resuming download to /Users/prakhar/.cache/kagglehub/datasets/gti-upm/leapgestrecog/1.archive (884998144/2286085038) bytes left.


100%|██████████| 2.13G/2.13G [02:08<00:00, 10.9MB/s]

Extracting files...


## Define Gesture Labels

Create labels for each gesture class.

In [3]:
gesture_labels = {
    "01_palm":0,
    "02_l":1,
    "03_fist":2,
    "04_fist_moved":3,
    "05_thumb":4,
    "06_index":5,
    "07_ok":6,
    "08_palm_moved":7,
    "09_c":8,
    "10_down":9
}

## Load and Preprocess Images

- Read images
- Resize images to 64 × 64
- Normalize pixel values
- Store images and labels

In [4]:
images = []
labels = []

people = sorted(os.listdir(dataset_path))

for person in people:

    person_path = os.path.join(dataset_path, person)

    gestures = sorted(os.listdir(person_path))

    for gesture in gestures:

        gesture_path = os.path.join(person_path, gesture)

        for filename in os.listdir(gesture_path):

            image_path = os.path.join(gesture_path, filename)

            image = cv2.imread(image_path)

            if image is None:
                continue

            image = cv2.resize(image,(64,64))

            image = image / 255.0

            images.append(image)

            labels.append(gesture_labels[gesture])

## Convert Data into NumPy Arrays

In [5]:
X = np.array(images,dtype=np.float32)
y = np.array(labels)

print(X.shape)
print(y.shape)

(20000, 64, 64, 3)
(20000,)


## One-Hot Encode Labels

In [6]:
y = to_categorical(y,num_classes=10)

## Split Dataset

Split the dataset into training and testing sets.

In [7]:
X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## Build CNN Model

Create the Convolutional Neural Network architecture.

In [8]:
model = Sequential()

model.add(Conv2D(
    32,
    (3,3),
    activation='relu',
    input_shape=(64,64,3)
))

model.add(MaxPooling2D(pool_size=(2,2)))

model.add(Conv2D(
    64,
    (3,3),
    activation='relu'
))

model.add(MaxPooling2D(pool_size=(2,2)))

model.add(Flatten())

model.add(Dense(
    128,
    activation='relu'
))

model.add(Dropout(0.5))

model.add(Dense(
    10,
    activation='softmax'
))

/Users/prakhar/CODE/Projects/SCT_ML_4/.venv/lib/python3.14/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


## Compile the Model

Configure the optimizer, loss function, and evaluation metric.

In [9]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## Train the Model

In [10]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step - accuracy: 0.8594 - loss: 0.4362 - val_accuracy: 0.9972 - val_loss: 0.0146
Epoch 2/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.9800 - loss: 0.0598 - val_accuracy: 0.9972 - val_loss: 0.0067
Epoch 3/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.9866 - loss: 0.0406 - val_accuracy: 0.9994 - val_loss: 0.0031
Epoch 4/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step - accuracy: 0.9930 - loss: 0.0232 - val_accuracy: 0.9984 - val_loss: 0.0067
Epoch 5/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - accuracy: 0.9898 - loss: 0.0294 - val_accuracy: 0.9994 - val_loss: 0.0026
Epoch 6/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - accuracy: 0.9930 - loss: 0.0199 - val_accuracy: 0.9991 - val_loss: 0.0036
Epoch 7/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - accuracy: 0.9948 - loss: 0.0145 - val_accuracy: 0.9997 - val_loss: 7.1429e-04
Epoch 8/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - accuracy: 0.9947 - loss: 0.0140 - val_

## Evaluate the Model

In [11]:
loss,accuracy = model.evaluate(
    X_test,
    y_test
)

print("Accuracy :",accuracy)

125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9998 - loss: 9.1067e-04
Accuracy : 0.999750018119812


## Make Predictions

In [12]:
prediction = model.predict(X_test)

predicted_class = np.argmax(prediction[0])

print("Predicted Gesture :",predicted_class)

125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Predicted Gesture : 3


# Conclusion

In this project, a Convolutional Neural Network (CNN) was developed to recognize different hand gestures using the LeapGestRecog dataset.

The images were preprocessed by resizing and normalizing them before training the model. The CNN successfully learned important features from the images and classified them into 10 different gesture classes.

This project helped in understanding image preprocessing, CNN architecture, model training, and gesture classification using deep learning.